# 第24讲：lora_svd_basics

原单元、历史输出及元数据保留；新增说明与代码注释不冒充原作者文字。教学CPU变体见复现记录。


### 原单元 0（markdown）

【建议会】原文链接Torch SVD及漂移的PEFT main行号。本次以torch2.5.1和PEFT0.12.0本地源码作可执行证据。


- torch svd
    - https://pytorch.org/docs/stable/generated/torch.linalg.svd.html
- lora merge by svd
    - https://github.com/huggingface/peft/blob/main/src/peft/tuners/lora/model.py#L622
    - https://github.com/huggingface/peft/blob/main/src/peft/tuners/lora/model.py#L674-L688

### 原单元 1（code）

【必须会】Image只负责展示，不是矩阵分解库。


In [21]:
from IPython.display import Image

### 原单元 2（code）

【必须会】远程LoRA图：原URL和HTML引用保留；未下载图片，未验证远端图像内容。新增解释提供可直接读的流程图。


In [23]:
Image(url='https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/peft/lora_diagram.png', width=500)

### 原单元 3（code）

【必须会】导入torch并固定seed0，影响后续随机矩阵。本次原版/注释版各自从该格重新设seed。


In [1]:
import torch
torch.random.manual_seed(0)

### 原单元 4（code）

【必须会】d,n分别是输出/输入维度100；low_rank20是构造与截断使用的目标秩，不是batch或token数。


In [2]:
# Define dimensions and low rank
d, n = 100, 100
low_rank = 20

### 原单元 5（code）

【必须会】两个随机float32矩阵[100,20]@[20,100]得到W[100,100]，数学秩至多20。随机实数下通常20，数值秩还受dtype和容差影响。


In [6]:
# Create a low rank matrix W
# lora_b @ lora_a
W = torch.randn(d, low_rank) @ torch.randn(low_rank, n)

### 原单元 6（code）

【必须会】matrix_rank使用奇异值容差判断数值秩。本次真实得到20，不把历史结果当新结果。


In [7]:
torch.linalg.matrix_rank(W)

tensor(20)

### 原单元 7（markdown）

【建议会】进入SVD，不做模型训练。


## svd

### 原单元 8（code）

【必须会】第二个远程图片展示，width400只影响HTML布局。


In [24]:
Image(url='https://www.statlect.com/images/matrix-product-and-rank__5.png', width=400)

### 原单元 9（markdown）

【建议会】低秩矩阵重建：保留20个主要奇异值应在浮点误差内恢复W。


### low rank svd

### 原单元 10（code）

【必须会】torch.linalg.svd(W,full_matrices=True)返回U[100,100]、S[100]、Vh[100,100]，Vh已是V的共轭转置。float32 CPU实际执行。


In [8]:
U, S, Vh = torch.linalg.svd(W, full_matrices=True)

### 原单元 11（code）

【必须会】只展示三个shape，方阵容易掩盖矩形full/reduced SVD尺寸差异，解释中另外推导。


In [10]:
U.shape, S.shape, Vh.shape

(torch.Size([100, 100]), torch.Size([100]), torch.Size([100, 100]))

### 原单元 12（code）

【必须会】diag把奇异值向量变对角矩阵；U@diag(S)@Vh重建W；torch.dist默认Frobenius距离，数值误差不是严格0。


In [11]:
torch.dist(U @ torch.diag(S) @ Vh, W)

tensor(0.0004)

### 原单元 13（code）

【必须会】切U前20列/S前20项/Vh前20行，再把S乘进U。变量U此后代表U20Σ20，不再是正交矩阵；重建用U@Vh，不能再乘一次S。


In [12]:
U = U[:, :low_rank]
S = S[:low_rank]
U = U @ torch.diag(S)
Vh = Vh[:low_rank, :]

### 原单元 14（code）

【必须会】比较截断重建误差。对构造的秩20矩阵应很小；不意味着任意100×100矩阵都能由秩20无损表示。


In [15]:
torch.dist(U @ Vh, W)

tensor(0.0004)

### 原单元 15（markdown）

【建议会】换成普通随机满秩矩阵，展示压缩误差。


### full rank svd

### 原单元 16（code）

【必须会】randn_like保持W的shape/device/dtype，重新生成独立随机W2，数学上几乎必然满秩。


In [17]:
W2 = torch.randn_like(W)

### 原单元 17（code）

【必须会】数值秩本次100；不是把low_rank变量改成100。


In [25]:
torch.linalg.matrix_rank(W2)

tensor(100)

### 原单元 18（code）

【必须会】对满秩W2做完整SVD，重新覆盖U/S/Vh三个变量。


In [18]:
U, S, Vh = torch.linalg.svd(W2, full_matrices=True)

### 原单元 19（code）

【必须会】完整100项重建W2应只有浮点误差，不能用此误差评价语言模型。


In [19]:
torch.dist(U @ torch.diag(S) @ Vh, W2)

tensor(0.0001)

### 原单元 20（code）

【必须会】只保留20项后有明显误差。最优秩20近似仍会丢弃剩余80项；其Frobenius误差平方等于被丢弃奇异值平方和。


In [20]:
U = U[:, :low_rank]
S = S[:low_rank]
U = U @ torch.diag(S)
Vh = Vh[:low_rank, :]

torch.dist(U @ Vh, W2)

tensor(67.6974)